# Sentence Share with the Centroid Vector

| Item | Description |
|---|---|
| Goal | The sentence-level relevance score (1A) with c = the statement centroid (T3.5.3): for each firm and sample, the share of its Item 1 sentences whose cosine similarity to c is at or above the threshold fixed in T3.6.5. |
| Input | `<scratch>/T3.6 - Firm Sentences/Processed/sentence_similarities.parquet` and `sentences.parquet`; `output/T3.4.1/filings.csv` (sample flags); `output/T3.6.5/similarity_threshold.csv` (the threshold for this vector). |
| Output | `output/T3.6.7/score_sentence_share_centroid.csv`: one row per firm and sample with the counts and the score; `output/T3.6.7/example_sentences_above_threshold.csv`: 20 random sentences at or above the threshold. |
| Unit | A firm (gvkey) within a sample; the sentence is the unit the share is counted over. |
| Sample | All years (18,779 filings, 1,058 firms) and fiscal years 2010, 2015, 2020 (2,072 filings, 841 firms), as flagged in T3.4.1. |
| Variables | • `n_sentences`: all Item 1 sentences of the firm's filings in the sample.<br>• `n_above_threshold`: those with $\cos(e_k, c) \ge \tau$, where $\tau$ is read from `similarity_threshold.csv`.<br>• `score`: $\text{score}_{is} = n\_above\_threshold / n\_sentences$, numerator and denominator summed over the firm's filings in sample $s$. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings

The threshold is read from the file T3.6.5 saved, never retyped here.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.6.7"  # the identifier names this notebook's output folder
PROCESSED_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Processed"
SIMILARITIES_FILE = PROCESSED_FOLDER / "sentence_similarities.parquet"
SENTENCES_FILE = PROCESSED_FOLDER / "sentences.parquet"
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
THRESHOLD_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.6.5/similarity_threshold.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

VECTOR = "centroid"
SIMILARITY_COLUMN = "sim_centroid"
SAMPLES = {"all_years": "in_all_years", "three_years": "in_three_years"}  # sample name to its flag in filings.csv
N_EXAMPLES = 20  # sentences shown from above the threshold
RANDOM_SEED = 0
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.7


## Step 3. Read the threshold

One row per vector in the T3.6.5 file; take this vector's threshold and the count it recorded, which the sentence table must reproduce.


In [3]:
threshold_table = pd.read_csv(THRESHOLD_FILE)
threshold_row = threshold_table.loc[threshold_table["vector"].eq(VECTOR)]
assert len(threshold_row) == 1
threshold_row = threshold_row.iloc[0]
SIMILARITY_THRESHOLD = float(threshold_row["threshold"])
print(f"Vector: {VECTOR}; threshold: {SIMILARITY_THRESHOLD}; decided by: {threshold_row['decided_by']}")
print(f"Reason: {threshold_row['reason']}")
print(f"Expected sentences at or above: {int(threshold_row['n_sentences_at_or_above']):,}")


Vector: centroid; threshold: 0.75; decided by: author, by hand
Reason: Reviewed the example sentences by similarity band and the share tables; chose 0.67 for the definition vector and 0.75 for the centroid.
Expected sentences at or above: 3,956


## Step 4. Load the sentences with their similarities

The similarity table and the sentence table are in the same row order; join them on the row number.


In [4]:
similarities = pd.read_parquet(SIMILARITIES_FILE, columns=["row", SIMILARITY_COLUMN])
sentences = pd.read_parquet(SENTENCES_FILE)
assert len(similarities) == len(sentences)
assert (similarities["row"].to_numpy() == np.arange(len(sentences))).all()
sentences["similarity"] = similarities[SIMILARITY_COLUMN].to_numpy()
del similarities
print(f"Sentences: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings")
display(sentences.head(3))


Sentences: 5,647,505 from 18,779 filings


,accessionNo,gvkey,fyear,sent_idx,sentence,similarity
0,0000912057-99-011120,001013,1999,0,"ADC Telecommunications, Inc. (""ADC"") was incor...",0.475178
1,0000912057-99-011120,001013,1999,1,"ADC offers a broad range of network equipment,...",0.503914
2,0000912057-99-011120,001013,1999,2,"ADC's broadband, multiservice network solution...",0.514493


## Step 5. Flag the sentences at or above the threshold

The count must equal the one T3.6.5 recorded.


In [5]:
sentences["above_threshold"] = (sentences["similarity"] >= SIMILARITY_THRESHOLD).astype(int)
n_above = int(sentences["above_threshold"].sum())
assert n_above == int(threshold_row["n_sentences_at_or_above"])
print(f"Sentences at or above {SIMILARITY_THRESHOLD}: {n_above:,} ({n_above / len(sentences):.3%})")


Sentences at or above 0.75: 3,956 (0.070%)


## Step 6. Load the filings with their sample flags and firm names

Every sentence's filing must be in the T3.4.1 list.


In [6]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
assert filings["accessionNo"].is_unique
sentences = sentences.merge(
    filings[["accessionNo", "conm", "in_all_years", "in_three_years"]], on="accessionNo", how="left", validate="many_to_one"
)
assert sentences["in_all_years"].notna().all()
print(f"Filings: {len(filings):,}; sentences matched: {len(sentences):,}")


Filings: 18,779; sentences matched: 5,647,505


## Step 7. Show 20 sentences that passed the threshold

A random draw from the sentences at or above the threshold, with the firm, fiscal year, and similarity, so the reader sees what the score counts. The draw is saved beside the scores.


In [7]:
rng = np.random.default_rng(RANDOM_SEED)
above = sentences.loc[sentences["above_threshold"].eq(1)]
examples = above.iloc[np.sort(rng.choice(len(above), size=N_EXAMPLES, replace=False))]
examples = examples[["gvkey", "conm", "fyear", "accessionNo", "sent_idx", "similarity", "sentence"]].copy()
examples["similarity"] = examples["similarity"].round(3)
examples.to_csv(OUTPUT_FOLDER / "example_sentences_above_threshold.csv", index=False)
pd.set_option("display.max_colwidth", 220)
display(examples[["conm", "fyear", "similarity", "sentence"]].reset_index(drop=True))


,conm,fyear,similarity,sentence
0,AMERICAN EXPRESS CO,2020,0.760,"Various regulators, U.S. states and territories are considering similar requirements or have adopted laws, rules and regulations pertaining to privacy and/or information and cyber security that may be more stringent ..."
1,AUTODESK INC,2013,0.750,Our inability to protect our proprietary information could harm our business.
2,IHEARTMEDIA INC,2021,0.785,"We are subject to a number of federal and state laws and regulations relating to consumer protection, information security, data protection and privacy."
3,FRANKLIN RESOURCES INC,2021,0.756,"A majority of the jurisdictions where we operate are covered, or we expect will be soon covered, by privacy and data protection laws and regulations."
4,UNITEDHEALTH GROUP INC,2015,0.793,Federal consumer protection laws may also apply in some instances to privacy and security practices related to personally identifiable information.
5,MORGAN STANLEY,2023,0.752,"Our businesses are also subject to increasing privacy and data protection legal requirements concerning the use and protection of certain personal information with regard to clients, employees and others."
6,MATCH GROUP INC,2024,0.766,"Because we receive, store, and use a substantial amount of information received from or generated by our users, we are particularly impacted by laws and regulations governing privacy; the storage, sharing, use, proce..."
7,BLOCK INC,2025,0.763,"Various legislative bodies and regulatory agencies in the United States and elsewhere in our international markets continue to examine a wide variety of issues that could impact our business, including privacy, data ..."
8,ALLSTATE CORP,2020,0.776,Privacy Regulation and Data Security.
9,CAPITAL ONE FINANCIAL CORP,2023,0.755,"For more information on privacy, data protection and data security requirements, please see “Privacy, Data Protection and Data Security.”"


## Step 8. Compute the score per firm and sample

For each sample, keep the firm's filings in that sample and sum the counts over them; the score is the ratio of the sums.


In [8]:
score_rows = []
for sample_name, flag_column in SAMPLES.items():
    in_sample = sentences.loc[sentences[flag_column].eq(1)]
    firm_counts = in_sample.groupby("gvkey").agg(
        conm=("conm", "first"),
        n_filings=("accessionNo", "nunique"),
        n_sentences=("sentence", "size"),
        n_above_threshold=("above_threshold", "sum"),
    ).reset_index()
    firm_counts["sample"] = sample_name
    firm_counts["score"] = firm_counts["n_above_threshold"] / firm_counts["n_sentences"]
    score_rows.append(firm_counts)
scores = pd.concat(score_rows, ignore_index=True)
scores["vector"] = VECTOR
scores["threshold"] = SIMILARITY_THRESHOLD
assert scores["score"].between(0, 1).all()
for sample_name in SAMPLES:
    part = scores.loc[scores["sample"].eq(sample_name)]
    print(f"{sample_name}: {len(part):,} firms, {int(part['n_filings'].sum()):,} filings; firms with a positive score: {int(part['score'].gt(0).sum()):,} ({part['score'].gt(0).mean():.1%}); median score among them {part.loc[part['score'].gt(0), 'score'].median():.4f}")
display(scores.head(3))


all_years: 1,058 firms, 18,779 filings; firms with a positive score: 265 (25.0%); median score among them 0.0016
three_years: 841 firms, 2,072 filings; firms with a positive score: 162 (19.3%); median score among them 0.0033


,gvkey,conm,n_filings,n_sentences,n_above_threshold,sample,score,vector,threshold
0,001013,ADC TELECOMMUNICATIONS INC,12,3786,0,all_years,0.000000,centroid,0.75
1,001045,AMERICAN AIRLINES GROUP INC,22,7288,3,all_years,0.000412,centroid,0.75
2,001075,PINNACLE WEST CAPITAL CORP,15,6823,0,all_years,0.000000,centroid,0.75


## Step 9. Show the ten firms with the highest score in each sample

A first look at who comes out on top; the figures in T3.9 do this properly.


In [9]:
for sample_name in SAMPLES:
    top = scores.loc[scores["sample"].eq(sample_name)].nlargest(10, "score")[["conm", "n_filings", "n_sentences", "n_above_threshold", "score"]]
    print(f"--- {sample_name}")
    display(top.reset_index(drop=True))


--- all_years


,conm,n_filings,n_sentences,n_above_threshold,score
0,META PLATFORMS INC,14,1870,35,0.018717
1,VERALTO CORP,3,554,9,0.016245
2,DUN & BRADSTREET HOLDNGS INC,5,1771,28,0.015810
3,TWITTER INC,9,1419,22,0.015504
4,TRADE DESK INC,10,2767,40,0.014456
5,BROADRIDGE FINANCIAL SOLUTNS,19,6630,87,0.013122
6,AMERICAN GREETINGS -CL A,17,1808,23,0.012721
7,PAYPAL HOLDINGS INC,11,2368,30,0.012669
8,BREAD FINANCIAL HOLDINGS INC,25,7244,91,0.012562
9,FORTIVE CORP,10,1737,21,0.012090


--- three_years


,conm,n_filings,n_sentences,n_above_threshold,score
0,META PLATFORMS INC,2,183,5,0.027322
1,FORTIVE CORP,1,203,4,0.019704
2,TWITTER INC,2,289,5,0.017301
3,AMERICAN GREETINGS -CL A,2,267,4,0.014981
4,DUN & BRADSTREET HOLDNGS INC,1,403,6,0.014888
5,BREAD FINANCIAL HOLDINGS INC,3,641,9,0.014041
6,BROADRIDGE FINANCIAL SOLUTNS,3,998,14,0.014028
7,GLOBAL PAYMENTS INC,3,649,8,0.012327
8,UNITEDHEALTH GROUP INC,3,1072,13,0.012127
9,EQUIFAX INC,3,981,11,0.011213


## Step 10. Save the scores

One row per firm and sample; the vector and threshold are repeated on every row so the file stands alone.


In [10]:
output_columns = ["gvkey", "conm", "sample", "vector", "threshold", "n_filings", "n_sentences", "n_above_threshold", "score"]
scores_file = OUTPUT_FOLDER / "score_sentence_share_centroid.csv"
scores[output_columns].sort_values(["sample", "gvkey"]).to_csv(scores_file, index=False)
print(f"Saved: {scores_file}")
print(f"Rows: {len(scores):,}; columns: {output_columns}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.7/score_sentence_share_centroid.csv
Rows: 1,899; columns: ['gvkey', 'conm', 'sample', 'vector', 'threshold', 'n_filings', 'n_sentences', 'n_above_threshold', 'score']
